# 04 — Semantic Search

## Goal

Replace the current lexical search approach with semantic search using embeddings.

This notebook will:

- Generate embeddings for document chunks.
- Generate an embedding for the user query.
- Compare the query with the document chunks.
- Retrieve the most semantically relevant results.
- Compare semantic search with the current MinSearch implementation.

## Why Semantic Search?

The current retrieval system relies mainly on lexical matching.

This works well when the query and the documents use similar words, but it may fail when they express the same idea using different vocabulary.

Semantic search represents text as numerical vectors called embeddings. Texts with similar meanings should appear close to each other in the vector space, even when they do not contain the same exact words.

## Retrieval Architecture

```text
User Question
      ↓
Query Embedding
      ↓
Vector Similarity Search
      ↓
Relevant Document Chunks
      ↓
Prompt Construction
      ↓
Large Language Model
      ↓
Answer

## Embedding Model Selection

For the semantic retrieval system, we will use OpenAI's `text-embedding-3-small` model.

This model converts text into numerical vectors called embeddings. Texts with similar meanings should produce vectors that are close to each other in the embedding space.

### Why this model?

- It provides strong semantic retrieval quality.
- It is easy to integrate using the OpenAI Python SDK.
- It is suitable for a portfolio-scale RAG application.
- It allows the retrieval architecture to remain independent from the language model used to generate the final answer.
- The embedding provider can be replaced later without redesigning the complete RAG pipeline.

### Architecture decision

Revenue AI Copilot will use two separate model providers:

- **OpenAI** for text embeddings.
- **Groq** for answer generation.

This separation allows each model to be selected according to its specific role.

In [54]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

## Generate Embeddings

In [55]:
response = client.embeddings.create(
    model="text-embedding-3-small",
    input="Revenue Management is the practice of selling the right room to the right customer at the right time."
)

embedding = response.data[0].embedding

print(type(embedding))
print(len(embedding))
print(embedding[:10])

<class 'list'>
1536
[-0.016021728515625, 0.002277374267578125, 0.046173095703125, 0.00862884521484375, 0.00777435302734375, -0.004131317138671875, -0.051361083984375, 0.0101318359375, 0.006740570068359375, -0.0025615692138671875]


In [56]:
def get_embedding(text, model="text-embedding-3-small"):
    response = client.embeddings.create(
        model=model,
        input=text
    )
    return response.data[0].embedding

In [57]:
test_embedding = get_embedding(
    "How can a hotel increase revenue during low-demand periods?"
)

print(type(test_embedding))
print(len(test_embedding))

<class 'list'>
1536


### Embedding Function

The `get_embedding()` function converts text into a 1536-dimensional numerical vector using OpenAI's `text-embedding-3-small` model.

Texts with similar meanings should produce vectors that are located close to each other in the embedding space.

In [58]:
text_1 = "Hotels can increase revenue by adjusting prices according to demand."
text_2 = "Dynamic pricing helps hotels optimize room rates based on market demand."
text_3 = "The hotel restaurant serves breakfast from 7:00 to 10:00."

In [59]:
embedding_1 = get_embedding(text_1)
embedding_2 = get_embedding(text_2)
embedding_3 = get_embedding(text_3)

## Cosine Similarity

In [60]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [61]:
similarity_12 = cosine_similarity(
    [embedding_1],
    [embedding_2]
)[0][0]

similarity_13 = cosine_similarity(
    [embedding_1],
    [embedding_3]
)[0][0]

print(f"Similarity 1-2: {similarity_12:.4f}")
print(f"Similarity 1-3: {similarity_13:.4f}")

Similarity 1-2: 0.7304
Similarity 1-3: 0.3198


In [62]:
from app.ingest import load_documents, create_chunks

source_documents = load_documents("data/raw")
chunks = create_chunks(source_documents)

print("Pages loaded:", len(source_documents))
print("Chunks created:", len(chunks))

print(chunks[0])

Pages loaded: 183
Chunks created: 358
{'source': 'Hotel Revenue Guide eBook_18.07.2023.pdf', 'page': 2, 'chunk_id': 0, 'text': '2. Hotel Revenue Managementwww.amadeus-hospitality.com Hotel revenue management basics A. What is hotel revenue management? B. What is the purpose of hotel revenue management? C. Key principles of hotel revenue management D. Importance of hotel revenue management E. Hotel revenue management key performance indicators (KPIs) F. Hotel revenue management origins Market segmentation by traveler type A. Capturing leisure demand B. Capturing business demand C. Capturing bleisure demand D. Capturing group business Pricing strategies for hotels A. What is hotel pricing optimization? B. Dynamic pricing strategies C. Differentiated pricing strategies Maximizing revenue opportunities: leisure, business, and group strategies A. Offer attractive add-ons B. Ensure rate parity C. Use the right distributi'}


## Generate Embeddings for Document Chunks

Each document chunk will be converted into an embedding and stored together with its metadata.

To make the process more efficient, embeddings will be generated in batches instead of sending one API request per chunk.

In [63]:
from time import sleep

EMBEDDING_MODEL = "text-embedding-3-small"
BATCH_SIZE = 50


def get_embeddings_batch(texts, model=EMBEDDING_MODEL):
    response = client.embeddings.create(
        model=model,
        input=texts
    )
    return [item.embedding for item in response.data]

In [64]:
semantic_documents = []

for start in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[start:start + BATCH_SIZE]
    batch_texts = [chunk["text"] for chunk in batch]

    batch_embeddings = get_embeddings_batch(batch_texts)

    for offset, (chunk, embedding) in enumerate(zip(batch, batch_embeddings)):
        semantic_documents.append({
            "id": start + offset,
            "source": chunk["source"],
            "page": chunk["page"],
            "chunk_id": chunk["chunk_id"],
            "text": chunk["text"],
            "embedding": embedding
        })

    print(
        f"Processed {min(start + BATCH_SIZE, len(chunks))}"
        f"/{len(chunks)} chunks"
    )

    sleep(0.2)

Processed 50/358 chunks
Processed 100/358 chunks
Processed 150/358 chunks
Processed 200/358 chunks
Processed 250/358 chunks
Processed 300/358 chunks
Processed 350/358 chunks
Processed 358/358 chunks


In [65]:
print("Semantic documents:", len(semantic_documents))
print("Embedding dimensions:", len(semantic_documents[0]["embedding"]))

print({
    key: value
    for key, value in semantic_documents[0].items()
    if key != "embedding"
})

Semantic documents: 358
Embedding dimensions: 1536
{'id': 0, 'source': 'Hotel Revenue Guide eBook_18.07.2023.pdf', 'page': 2, 'chunk_id': 0, 'text': '2. Hotel Revenue Managementwww.amadeus-hospitality.com Hotel revenue management basics A. What is hotel revenue management? B. What is the purpose of hotel revenue management? C. Key principles of hotel revenue management D. Importance of hotel revenue management E. Hotel revenue management key performance indicators (KPIs) F. Hotel revenue management origins Market segmentation by traveler type A. Capturing leisure demand B. Capturing business demand C. Capturing bleisure demand D. Capturing group business Pricing strategies for hotels A. What is hotel pricing optimization? B. Dynamic pricing strategies C. Differentiated pricing strategies Maximizing revenue opportunities: leisure, business, and group strategies A. Offer attractive add-ons B. Ensure rate parity C. Use the right distributi'}


## Semantic Search

The semantic search function converts the user query into an embedding, compares it with all document embeddings using cosine similarity, and returns the most relevant chunks together with their metadata and similarity scores.

In [66]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


def embed_query(query, model=EMBEDDING_MODEL):
    response = client.embeddings.create(
        model=model,
        input=query
    )
    return response.data[0].embedding

In [67]:
def search_semantic(query, top_k=5):
    query_embedding = embed_query(query)

    document_embeddings = np.array([
        doc["embedding"]
        for doc in semantic_documents
    ])

    similarities = cosine_similarity(
        [query_embedding],
        document_embeddings
    )[0]

    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = []

    for index in top_indices:
        doc = semantic_documents[index]

        results.append({
            "id": doc["id"],
            "source": doc["source"],
            "page": doc["page"],
            "chunk_id": doc["chunk_id"],
            "text": doc["text"],
            "score": float(similarities[index])
        })

    return results

In [68]:
query = "How can hotels improve revenue during periods of low demand?"

results = search_semantic(query, top_k=5)

for result in results:
    print(f"Score: {result['score']:.4f}")
    print(f"Source: {result['source']}")
    print(f"Page: {result['page']}")
    print(result["text"][:500])
    print("-" * 80)

Score: 0.6827
Source: Beginners_Guide_to_Revenue_Management.pdf
Page: 11
by closing the more expensive (and least profitable) channels when demand and booking pace is high. Then sit back and watch your ADR go up during the high demand periods, which leads to a proportional increase in your profits. The ultimate guide to hotel revenue management 11
--------------------------------------------------------------------------------
Score: 0.6523
Source: Revenue-Management-Manual-Xotels-2.pdf
Page: 5
t so that you can be proactive and not reactive. Use the information to divide your market and adjust your products through distribution, to the right customer at the right time and at the right price. Revenue Management is not only maximizing in high period demand, it helps stimulating demand in low periods while avoiding pricing cannibalism. Revenue Management is long term strategic, takes all revenue with their profitability into consideration, can sell low rates even in high demand period. W

In [69]:
query = """
Hotel revenue management strategies specifically for low-demand periods:
stimulating demand, increasing occupancy, promotions, packages,
discounts, segmentation and pricing during weak demand.
"""

results = search_semantic(query, top_k=5)

for result in results:
    print(f"Score: {result['score']:.4f}")
    print(f"Source: {result['source']}")
    print(f"Page: {result['page']}")
    print(result["text"][:500])
    print("-" * 80)

Score: 0.6953
Source: Hotel Revenue Guide eBook_18.07.2023.pdf
Page: 12
re a memorable experience. • Try out new offerings frequently to experiment and identify which ones are most successful. The most effective hotel revenue management strategies not only require flexible pricing but also differentiation of your hotel offerings outside of the guest room. As you think about updating or creating new value added promotions, first consider: • The booking window: To maximize your profitability, continuously monitor the booking window across your most profitable segments
--------------------------------------------------------------------------------
Score: 0.6843
Source: Revenue-Management-Manual-Xotels-2.pdf
Page: 5
t so that you can be proactive and not reactive. Use the information to divide your market and adjust your products through distribution, to the right customer at the right time and at the right price. Revenue Management is not only maximizing in high period demand, it helps s

## Semantic RAG

The semantic RAG pipeline retrieves relevant chunks using embeddings, builds a context from those chunks, and sends the context to the Groq-hosted language model.

The answer should be based only on the retrieved documents and should include source references.

In [70]:
from openai import OpenAI
import os

groq_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [71]:
SEMANTIC_RAG_INSTRUCTIONS = """
You are Revenue AI Copilot, an assistant specialized in hotel revenue management.

Answer the user's question using only the provided context.

Requirements:
- Give a clear and practical answer.
- Do not invent information that is not present in the context.
- If the context is insufficient, say so.
- Mention the supporting source and page for every main recommendation.
- Prefer context directly related to the user's exact situation.
- Ignore context that refers to the opposite situation or contradicts the question.
- For questions about low demand, do not use recommendations that apply only to high-demand periods.
"""

In [72]:
def expand_retrieval_query(query):
    return f"""
{query}

Focus specifically on:
low demand periods, weak demand, low occupancy, demand stimulation,
promotions, packages, segmentation, discounts, value-added offers,
pricing strategies and attracting additional bookings.

Exclude strategies that apply only to high-demand periods.
""".strip()

In [73]:
def build_semantic_context(search_results):
    context_parts = []

    for result in search_results:
        context_parts.append(
            f"""
Source: {result["source"]}
Page: {result["page"]}
Similarity score: {result["score"]:.4f}
Text: {result["text"]}
""".strip()
        )

    return "\n\n---\n\n".join(context_parts)

In [74]:
def semantic_llm(user_prompt, model="llama-3.1-8b-instant"):
    response = groq_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SEMANTIC_RAG_INSTRUCTIONS},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0
    )

    return response.choices[0].message.content

In [75]:
def rag_semantic(query, top_k=5):
    retrieval_query = expand_retrieval_query(query)
    search_results = search_semantic(retrieval_query, top_k=top_k)

    context = build_semantic_context(search_results)

    user_prompt = f"""
Question:
{query}

Context:
{context}
""".strip()

    answer = semantic_llm(user_prompt)

    return {
        "answer": answer,
        "retrieved_sources": [
            {
                "source": result["source"],
                "page": result["page"],
                "score": result["score"]
            }
            for result in search_results
        ]
    }

In [76]:
result = rag_semantic(
    "How can hotels improve revenue during periods of low demand?",
    top_k=5
)

print(result["answer"])

print("\nRetrieved sources:")
for source in result["retrieved_sources"]:
    print(
        f'- {source["source"]}, page {source["page"]}, '
        f'score {source["score"]:.4f}'
    )

To improve revenue during periods of low demand, hotels can consider the following strategies:

1. **Monitor the booking window**: Continuously monitor the booking window across your most profitable segments to identify any trends that point toward demand growth (Hotel Revenue Guide eBook, Page 12).
2. **Gather market data**: Collect current and forward-looking occupancy, ADR, and RevPAR data about your competitive sets to create new KPIs and understand how aggressive you need to be to attract new business (Hotel Revenue Guide eBook, Page 20).
3. **Streamline RFP responses and flexible group contracts**: Enhance efficiency by promptly responding to RFPs using integrated proposal templates and incorporate flexibility into your group contracts based on your hotel's capability and operational capacities (Hotel Revenue Guide eBook, Page 16).
4. **Create customized packages**: Offer tailored group packages that cater to the specific needs of different types of groups, such as corporate meet

## Semantic RAG Result

The semantic RAG pipeline successfully retrieves relevant chunks from multiple Revenue Management documents and generates a contextual answer with source and page references.

The experiment also shows that semantic similarity alone does not guarantee exact relevance. Query expansion and clear LLM instructions improve the final answer, while future versions should include hybrid retrieval, re-ranking, and more precise source attribution.